# 29 — Quantum Fisher Specification

**Repo:** `decoherence-free-sensing`  
**Notebook role:** information-theoretic precision specification  
**Previous:** `23_lieb_mattis_states.ipynb`  
**Next:** `37_two_body_measurement.ipynb`

Notebook 23 established admissible decoherence-free sensing states. This notebook shows how retained differential generator variance specifies Quantum Fisher Information, which in turn specifies the achievable precision of differential sensing.

For a pure state and differential phase generator \(J_z^-\),

\[
F_Q[|\psi\rangle,J_z^-]=4\,\mathrm{Var}_\psi(J_z^-).
\]

The quantum Cramér-Rao bound gives

\[
\Delta^2\varphi \geq \frac{1}{F_Q}.
\]

For the Lieb-Mattis scaling reference, we use

\[
\Delta^2\varphi_{\rm LM}
\sim
\frac{3}{N^2+4N}.
\]

## Engineering statement

The DFS constraint specifies admissible states; retained differential generator variance specifies Quantum Fisher Information; Quantum Fisher Information specifies achievable precision.


## Notebook outputs

Running this notebook creates:

- `results/figures/29_quantum_fisher_specification_v3.png`
- `results/figures/29_generator_variance_specifies_qfi_v3.png`
- `results/figures/29_quantum_precision_limits_v3.png`
- `results/figures/29_precision_gain_over_sql_v3.png`
- `results/figures/29_quantum_fisher_specification_pipeline_v3.png`
- `results/qfi_specification_summary.csv`
- `results/qfi_specification_summary.json`
- `results/decoherence_free_sensing_qfi_specification_outputs.zip`


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

ROOT = Path.cwd()
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
SITE = ROOT / "site"

for path in [RESULTS, FIGURES, SITE]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
})


## 1. Quantum Fisher specification

A decoherence-free constraint alone does not determine sensing performance.

Useful sensing requires retained differential generator variance. That retained variance specifies the Quantum Fisher Information,

\[
F_Q = 4\,\mathrm{Var}(J_z^-),
\]

which specifies the Quantum Cramér-Rao precision bound

\[
\Delta^2\varphi \geq \frac{1}{F_Q}.
\]

The engineering objective is therefore not merely to remain inside the DFS, but to maximize admissible differential generator variance.


In [ ]:
def draw_box(ax, xy, width, height, title, body=None, fontsize=15, facecolor="white", edgecolor="black", lw=1.8):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.025,rounding_size=0.04",
        linewidth=lw,
        edgecolor=edgecolor,
        facecolor=facecolor
    )
    ax.add_patch(box)
    ax.text(
        x + width/2,
        y + height*0.62 if body else y + height/2,
        title,
        ha="center",
        va="center",
        fontsize=fontsize,
        fontweight="bold"
    )
    if body:
        ax.text(
            x + width/2,
            y + height*0.28,
            body,
            ha="center",
            va="center",
            fontsize=11
        )

def draw_arrow(ax, p0, p1):
    ax.add_patch(FancyArrowPatch(
        p0, p1,
        arrowstyle="-|>",
        mutation_scale=18,
        linewidth=1.8
    ))

fig, ax = plt.subplots(figsize=(13.0, 5.8))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.92, "Quantum Fisher Specification", ha="center", va="center", fontsize=23)

items = [
    (0.05, "Constrain", "fix $J_z^+$\nDFS admissible"),
    (0.285, "Retain", "spread in\n$J_z^-$"),
    (0.52, "Quantify", "$F_Q = 4 Var$"),
    (0.755, "Compare", "SQL / HL /\nLieb-Mattis"),
]
w, h, y = 0.19, 0.32, 0.42

for x0, title, body in items:
    draw_box(ax, (x0, y), w, h, title, body)

for x0 in [0.245, 0.48, 0.715]:
    draw_arrow(ax, (x0, y + h/2), (x0 + 0.035, y + h/2))

ax.text(
    0.5,
    0.22,
    "A DFS constraint specifies admissible states; retained differential variance specifies Quantum Fisher Information.",
    ha="center",
    va="center",
    fontsize=14
)

path = FIGURES / "29_quantum_fisher_specification_v3.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 2. Generator variance specifies QFI

Notebook 23 established admissible DFS states. Notebook 29 shows that the retained spread of the differential generator directly specifies the available Quantum Fisher Information.

For a pure state under a unitary phase shift,

\[
|\psi_\varphi\rangle
=
e^{-i\varphi G}|\psi\rangle,
\]

the quantum Fisher information is

\[
F_Q = 4\,\mathrm{Var}_\psi(G).
\]

For differential sensing,

\[
G=J_z^-.
\]

After the DFS constraint removes the common coordinate, the remaining resource is the variance of the differential generator.


In [ ]:
var_values = np.linspace(0, 25, 100)
qfi_values = 4 * var_values

fig, ax = plt.subplots(figsize=(8.4, 5.8))
ax.plot(var_values, qfi_values, linewidth=2.2, label=r"$F_Q = 4\,\mathrm{Var}(J_z^-)$")

ax.set_title("Generator Variance Specifies Quantum Fisher Information")
ax.set_xlabel(r"differential generator variance $\mathrm{Var}(J_z^-)$")
ax.set_ylabel(r"quantum Fisher information $F_Q$")
ax.grid(True, alpha=0.3)
ax.legend()

ax.text(
    0.58,
    0.22,
    r"larger $\mathrm{Var}(J_z^-)$" + "\n" + r"$\Rightarrow$ larger $F_Q$",
    transform=ax.transAxes,
    ha="center",
    va="center",
    fontsize=13,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9)
)

path = FIGURES / "29_generator_variance_specifies_qfi_v3.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 3. Quantum precision limits

We compare three variance references for total atom number \(N\):

Standard quantum limit:

\[
\Delta^2\varphi_{\rm SQL}\sim \frac{1}{N}.
\]

Heisenberg reference:

\[
\Delta^2\varphi_{\rm HL}\sim \frac{1}{N^2}.
\]

Lieb-Mattis DFS-compatible reference:

\[
\Delta^2\varphi_{\rm LM}
\sim
\frac{3}{N^2+4N}.
\]

The Lieb-Mattis curve approaches Heisenberg-like scaling up to a constant factor while respecting the DFS constraint.


In [ ]:
N = np.unique(np.round(np.logspace(np.log10(4), np.log10(2000), 220)).astype(int))

sql = 1 / N
hl = 1 / (N**2)
lm = 3 / (N**2 + 4*N)

fig, ax = plt.subplots(figsize=(8.6, 6.0))

ax.loglog(N, sql, linewidth=2.2, label="SQL: 1/N")
ax.loglog(N, hl, linewidth=2.2, label="Heisenberg: 1/N²")
ax.loglog(N, lm, linewidth=2.2, label="Lieb-Mattis QCRB: 3/(N²+4N)")

ax.set_title("Quantum Precision Limits")
ax.set_xlabel("total atom number N")
ax.set_ylabel(r"variance reference $\Delta^2\varphi$")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

# Asymptotic slope labels.
ax.text(9, 0.08, "SQL\nO(N⁻¹)", fontsize=11)
ax.text(35, 0.0008, "Heisenberg\nO(N⁻²)", fontsize=11)
ax.text(150, 0.00006, "Lieb-Mattis\nO(N⁻²)\nconstant-factor gap", fontsize=11)

path = FIGURES / "29_quantum_precision_limits_v3.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 4. Precision gain over SQL

The SQL reference is a baseline:

\[
\Delta^2\varphi_{\rm SQL}=\frac{1}{N}.
\]

The Lieb-Mattis gain over SQL is

\[
\frac{\Delta^2\varphi_{\rm SQL}}
{\Delta^2\varphi_{\rm LM}}
=
\frac{N^2+4N}{3N}
=
\frac{N+4}{3}.
\]

So the DFS-compatible Lieb-Mattis reference improves linearly over SQL while retaining common-mode rejection.


In [ ]:
gain_lm_over_sql = sql / lm
gain_hl_over_sql = sql / hl

fig, ax = plt.subplots(figsize=(8.4, 5.8))

ax.loglog(N, gain_lm_over_sql, linewidth=2.2, label="Lieb-Mattis gain over SQL")
ax.loglog(N, gain_hl_over_sql, linestyle="--", linewidth=2.2, label="Heisenberg gain over SQL")

ax.set_title("Precision Gain Over SQL")
ax.set_xlabel("total atom number N")
ax.set_ylabel(r"gain: $\Delta^2\varphi_{\rm SQL}/\Delta^2\varphi$")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

ax.text(
    0.52,
    0.34,
    "linear gain over SQL\n" + r"$\approx (N+4)/3$",
    transform=ax.transAxes,
    ha="center",
    va="center",
    fontsize=13,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9)
)

path = FIGURES / "29_precision_gain_over_sql_v3.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 5. Quantum Fisher specification pipeline

The specification chain is:

\[
J_z^+=0
\quad\Rightarrow\quad
\mathrm{Var}(J_z^-)
\quad\Rightarrow\quad
F_Q
\quad\Rightarrow\quad
\Delta^2\varphi.
\]

Notebook 23 specified useful DFS states. Notebook 29 quantifies them.


In [ ]:
fig, ax = plt.subplots(figsize=(13.0, 5.8))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.92, "Quantum Fisher Specification Pipeline", ha="center", va="center", fontsize=23)

items = [
    (0.05, "DFS Constraint", "$J_z^+ = 0$"),
    (0.285, "Differential\nGenerator", "$J_z^-$"),
    (0.52, "Quantum Fisher\nInformation", "$F_Q = 4Var(J_z^-)$"),
    (0.755, "Precision\nBound", "Delta^2 phi >= 1/F_Q"),
]
w, h, y = 0.19, 0.32, 0.42

for x0, title, body in items:
    draw_box(ax, (x0, y), w, h, title, body, fontsize=14)

for x0 in [0.245, 0.48, 0.715]:
    draw_arrow(ax, (x0, y + h/2), (x0 + 0.035, y + h/2))

ax.text(
    0.5,
    0.22,
    "Retained differential sensitivity specifies achievable precision through Quantum Fisher Information.",
    ha="center",
    va="center",
    fontsize=14
)

path = FIGURES / "29_quantum_fisher_specification_pipeline_v3.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 6. Engineering questions

| Quantity | Specifies | Engineering meaning |
|---|---|---|
| \(J_z^+=0\) | admissible DFS sector | reject common-mode noise |
| \(\mathrm{Var}(J_z^-)\) | differential spread | retained sensing resource |
| \(F_Q\) | achievable information | precision resource |
| QCRB | minimum variance | measurement limit |
| Lieb-Mattis scaling | DFS-compatible gain | near-Heisenberg scaling up to a constant factor |


In [ ]:
summary = pd.DataFrame([
    {
        "quantity": "J_z^+ = 0",
        "specifies": "admissible DFS sector",
        "engineering_meaning": "reject common-mode noise"
    },
    {
        "quantity": "Var(J_z^-)",
        "specifies": "differential spread",
        "engineering_meaning": "retained sensing resource"
    },
    {
        "quantity": "F_Q",
        "specifies": "achievable information",
        "engineering_meaning": "precision resource"
    },
    {
        "quantity": "QCRB",
        "specifies": "minimum variance",
        "engineering_meaning": "measurement limit"
    },
    {
        "quantity": "Delta^2 varphi_LM = 3/(N^2+4N)",
        "specifies": "DFS-compatible scaling",
        "engineering_meaning": "near-Heisenberg scaling up to a constant factor"
    },
    {
        "quantity": "SQL/LM = (N+4)/3",
        "specifies": "gain over SQL",
        "engineering_meaning": "linear precision gain while respecting the DFS constraint"
    },
])

summary_csv = RESULTS / "qfi_specification_summary.csv"
summary_json = RESULTS / "qfi_specification_summary.json"

summary.to_csv(summary_csv, index=False)
summary.to_json(summary_json, orient="records", indent=2)

summary


## 7. Conclusion

The DFS constraint specifies admissible states; retained differential generator variance specifies Quantum Fisher Information; Quantum Fisher Information specifies achievable precision.

Notebook 29 establishes the information-theoretic limit of an admissible DFS state.

Notebook 37 asks the complementary engineering question: how should measurements be designed to extract that information while preserving the differential sensitivity established here?


## 8. Export notebook outputs

This cell creates one zip archive containing all generated figures and summary files.


In [ ]:
zip_path = RESULTS / "decoherence_free_sensing_qfi_specification_outputs.zip"

public_figures = [
    FIGURES / "29_quantum_fisher_specification_v3.png",
    FIGURES / "29_generator_variance_specifies_qfi_v3.png",
    FIGURES / "29_quantum_precision_limits_v3.png",
    FIGURES / "29_precision_gain_over_sql_v3.png",
    FIGURES / "29_quantum_fisher_specification_pipeline_v3.png",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in public_figures:
        if file.exists():
            zf.write(file, file.relative_to(ROOT))
    zf.write(summary_csv, summary_csv.relative_to(ROOT))
    zf.write(summary_json, summary_json.relative_to(ROOT))

print(f"Created: {zip_path}")

# Optional Colab download:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

zip_path


## 9. Next notebook

Suggested next notebook:

`37_two_body_measurement.ipynb`

Core goal:

- connect the QFI scaling reference to an observable measurement,
- identify the two-body fringe response,
- show why the operating point \(\varphi=\pi/4\) maximizes slope,
- translate state sensitivity into a measurement protocol.
